# Trend-Filtered Basis Arbitrage: WBTC vs. BTC (Wrapped Bitcoin Basis)

Third pair in this trend + mean-reversion combination series: **WBTC** (Wrapped Bitcoin — a custodial,
BitGo-backed ERC-20 token redeemable 1:1 for BTC, but only through a KYC'd merchant/custodian
process, not a fully open on-chain mechanism) vs. **BTC** itself.

**Why this pair, and a real caveat up front**: WBTC is the dominant way BTC gets used as DeFi
collateral, and that collateral demand is exactly the kind of thing that could create a trend-linked
basis — more leveraged borrowing against WBTC during BTC uptrends (chasing the rally) could bid WBTC
to a premium, while deleveraging/liquidations during downtrends could push it to a discount. WBTC has
also had real, documented **trust/custody events**: in mid-2023 BitGo's sole-custodian role was
restructured to include Justin Sun-affiliated entities, which caused real depeg concern and at least
one major exchange (**Coinbase — confirmed by checking their API directly: `WBTC-BTC` and `WBTC-USD`
are both listed as `"status": "delisted"`, `"trading_disabled": true`**) to drop WBTC pairs entirely.
That's directly relevant color for why a basis might exist here — but our data source (Kraken, since
Coinbase no longer carries this pair) only goes back to **August 2024**, which is *after* that 2023
custody controversy, similar to how the stETH notebook's sample postdates the 2022 depeg crisis.

**Hypothesis, stated directionally**: `trend_pct` and `basis_z` should be **positively correlated** —
BTC uptrends coinciding with WBTC richening (leveraged collateral demand), downtrends coinciding with
WBTC cheapening (deleveraging/redemption pressure) — the same directional logic as the stETH notebook,
applied to a custodial-wrap mechanism instead of a liquid-staking one.

**Data source**: Kraken's public API (free, no key). `WBTCXBT` gives WBTC's exchange rate directly in
BTC terms — same clean-cross-rate approach as `STETH-ETH` in the companion notebook. One real
limitation: Kraken's OHLC endpoint serves at most ~720 recent daily candles per call and isn't
further paginable for deeper history through this free endpoint — confirmed empirically (varying the
`since` parameter to a very old timestamp still returns the same ~720-candle window), so **~2 years
is the practical ceiling** for this specific approach, not a sign the pair is younger than that.

> ⚠️ Research only, not investment advice. Models a market-neutral spread position (long one leg,
> short the other) with no transaction costs, WBTC merchant-custody minimums, or execution slippage.


In [ ]:
# If needed, install dependencies (uncomment to run):
# %pip install pandas numpy matplotlib seaborn scipy statsmodels requests

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

pd.set_option("display.float_format", lambda x: f"{x:,.5f}")
sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)


## 1. Data Collection

`WBTCXBT` (WBTC's exchange rate directly in BTC terms) and `XBTUSD` (BTC's own $ price, for the
trend signal and for a $ price chart) — both from Kraken's public OHLC endpoint.


In [ ]:
KRAKEN_BASE = "https://api.kraken.com/0/public/OHLC"


def fetch_kraken_ohlc(pair, interval=1440):
    r = requests.get(KRAKEN_BASE, params={"pair": pair, "interval": interval}, timeout=20)
    r.raise_for_status()
    payload = r.json()
    if payload.get("error"):
        raise RuntimeError(payload["error"])
    result_key = [k for k in payload["result"] if k != "last"][0]
    rows = payload["result"][result_key]
    df = pd.DataFrame(rows, columns=["ts", "open", "high", "low", "close", "vwap", "volume", "count"])
    df["date"] = pd.to_datetime(df["ts"].astype(np.int64), unit="s").dt.normalize()
    df = df.drop_duplicates(subset="date").set_index("date").sort_index()
    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = df[c].astype(float)
    return df[["open", "high", "low", "close", "volume"]]


wbtc_xbt = fetch_kraken_ohlc("WBTCXBT")
xbt_usd = fetch_kraken_ohlc("XBTUSD")

print(f"WBTCXBT: {wbtc_xbt.index.min().date()} to {wbtc_xbt.index.max().date()}, {len(wbtc_xbt)} days")
print(f"XBTUSD:  {xbt_usd.index.min().date()} to {xbt_usd.index.max().date()}, {len(xbt_usd)} days")


## 2. Basis Construction

`WBTCXBT`'s close price is the exchange rate directly — 1.0 means WBTC trades at exact parity with
BTC.


In [ ]:
df = pd.DataFrame({
    "wbtc_btc_rate": wbtc_xbt["close"],
    "btc_usd": xbt_usd["close"],
}).dropna()

df["basis_pct"] = df["wbtc_btc_rate"] - 1

print(f"Overlapping sample: {df.index.min().date()} to {df.index.max().date()}, {len(df)} days")
df.tail()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(df.index, df["btc_usd"], color="tab:blue")
axes[0].set_title("BTC-USD Price")

axes[1].plot(df.index, df["basis_pct"] * 100, color="tab:green")
axes[1].axhline(0, color="gray", linewidth=0.8)
axes[1].set_title("WBTC/BTC Basis: (WBTCXBT rate - 1) x 100%")
axes[1].set_ylabel("%")
plt.tight_layout()
plt.show()

print(f"Basis stats: mean={df['basis_pct'].mean()*100:.4f}%, std={df['basis_pct'].std()*100:.4f}%, "
      f"min={df['basis_pct'].min()*100:.4f}%, max={df['basis_pct'].max()*100:.4f}%")


In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_stat, p_value, *_ = adfuller(df["basis_pct"].dropna())
print(f"ADF test on basis_pct: statistic={adf_stat:.3f}, p-value={p_value:.4f}")
print("p < 0.05 suggests the basis is stationary (mean-reverting) over this sample.")


## 3. Trend Detection (on BTC)


In [ ]:
def find_pivots(price, window=5):
    is_high = price == price.rolling(window * 2 + 1, center=True).max()
    is_low = price == price.rolling(window * 2 + 1, center=True).min()
    return price[is_high], price[is_low]


def fit_trendline(pivot_points, n_recent=3):
    if len(pivot_points) < 2:
        return None
    recent = pivot_points.iloc[-n_recent:]
    x = (recent.index - recent.index[0]).days.values.astype(float)
    y = recent.values
    slope, intercept = np.polyfit(x, y, 1)
    return recent.index[0], slope, intercept


swing_highs, swing_lows = find_pivots(df["btc_usd"], window=5)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(df.index, df["btc_usd"], color="tab:blue", label="BTC-USD close", zorder=1)
ax.scatter(swing_highs.index, swing_highs.values, color="red", s=25, zorder=3, label="swing highs")
ax.scatter(swing_lows.index, swing_lows.values, color="green", s=25, zorder=3, label="swing lows")

for pivots, color in [(swing_highs, "red"), (swing_lows, "green")]:
    fit = fit_trendline(pivots, n_recent=4)
    if fit:
        x0, slope, intercept = fit
        x_line = pd.date_range(x0, df.index[-1])
        y_line = intercept + slope * (x_line - x0).days
        ax.plot(x_line, y_line, color=color, linestyle="--", linewidth=1.5, alpha=0.8)

ax.set_title("BTC Price with Recent Swing-High / Swing-Low Trend Lines")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
def rolling_trend_pct(price, window=20):
    def slope(y):
        x = np.arange(len(y))
        return np.polyfit(x, y, 1)[0]
    raw_slope = price.rolling(window).apply(slope, raw=True)
    return raw_slope * window / price

df["trend_pct"] = rolling_trend_pct(df["btc_usd"], window=20)

fig, ax = plt.subplots()
df["trend_pct"].mul(100).plot(ax=ax, color="purple")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title("Rolling 20-Day BTC Trend Signal (Implied %% Move Over Window)")
ax.set_ylabel("%")
plt.tight_layout()
plt.show()


## 4. Does BTC's Trend Predict the WBTC Basis?


In [ ]:
def zscore(s, window=20):
    return (s - s.rolling(window).mean()) / s.rolling(window).std()

df["basis_z"] = zscore(df["basis_pct"], window=20)

sub = df[["trend_pct", "basis_z"]].dropna()
corr_val = sub["trend_pct"].corr(sub["basis_z"])

fig, ax = plt.subplots()
ax.scatter(sub["trend_pct"] * 100, sub["basis_z"], alpha=0.5, s=20)
ax.set_xlabel("Rolling BTC trend signal (%% move over 20d)")
ax.set_ylabel("WBTC basis z-score")
ax.set_title(f"BTC Trend vs. WBTC Basis Deviation (n={len(sub)}, corr={corr_val:.3f})")
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
plt.tight_layout()
plt.show()

print(f"Correlation(trend_pct, basis_z) = {corr_val:.3f}")
if abs(corr_val) < 0.15:
    print("Near zero: no meaningful relationship in this sample — the collateral-demand hypothesis "
          "isn't showing up here. A trend filter wouldn't be expected to help.")
elif corr_val > 0:
    print("Positive, as the collateral-demand hypothesis predicts: BTC uptrends coincide with WBTC "
          "richening, downtrends with WBTC cheapening.")
else:
    print("Negative — the opposite of the collateral-demand hypothesis. Worth investigating why "
          "before leaning on it.")


## 5. Mean-Reversion Arbitrage Signal

**Position convention**: `+1` = long the basis (long WBTC / short BTC, taken when WBTC is *cheap*
relative to its own recent history). `-1` = short the basis (short WBTC / long BTC, taken when WBTC
is unusually rich).


In [ ]:
def build_positions(z, entry=2.0, exit=0.5, blocked=None):
    position = pd.Series(0.0, index=z.index)
    current = 0.0
    for i, (idx, val) in enumerate(z.items()):
        if pd.isna(val):
            position.iloc[i] = current
            continue
        is_blocked = bool(blocked.loc[idx]) if blocked is not None and idx in blocked.index else False
        if current == 0.0:
            if not is_blocked:
                if val > entry:
                    current = -1.0
                elif val < -entry:
                    current = 1.0
        else:
            if abs(val) < exit:
                current = 0.0
        position.iloc[i] = current
    return position


ENTRY_Z, EXIT_Z = 2.0, 0.5
position_pure_mr = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
df["basis_z"].plot(ax=axes[0], color="tab:green")
axes[0].axhline(ENTRY_Z, color="red", linestyle="--", linewidth=0.8)
axes[0].axhline(-ENTRY_Z, color="red", linestyle="--", linewidth=0.8)
axes[0].axhline(0, color="gray", linewidth=0.8)
axes[0].set_title("WBTC Basis Z-Score with Entry Thresholds")

position_pure_mr.plot(ax=axes[1], color="black", drawstyle="steps-post")
axes[1].set_title("Pure Mean-Reversion Position (+1 = long basis, -1 = short basis)")
plt.tight_layout()
plt.show()


## 6. Combining Both Ideas: Trend-Filtered Arbitrage


In [ ]:
trend_strength = df["trend_pct"].abs()
trend_threshold = trend_strength.rolling(90, min_periods=30).quantile(0.67)
strong_trend = trend_strength > trend_threshold

position_trend_filtered = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z, blocked=strong_trend)

print(f"Days blocked by trend filter: {strong_trend.sum()} / {strong_trend.notna().sum()} "
      f"({strong_trend.mean()*100:.1f}%)")


In [ ]:
def sharpe(returns, periods_per_year=365):
    r = returns.dropna()
    if len(r) == 0 or r.std() == 0:
        return np.nan
    return (r.mean() / r.std()) * np.sqrt(periods_per_year)


def max_drawdown(cum_returns):
    running_max = cum_returns.cummax()
    dd = cum_returns / running_max - 1
    return dd.min()


basis_chg = df["basis_pct"].diff()

strategies = pd.DataFrame({
    "pure_mean_reversion": position_pure_mr.shift(1) * basis_chg,
    "trend_filtered_mean_reversion": position_trend_filtered.shift(1) * basis_chg,
})

cum = (1 + strategies.fillna(0)).cumprod()
cum.plot(title="Cumulative Return: Pure vs. Trend-Filtered WBTC/BTC Basis Arbitrage")
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    "annualized_sharpe": strategies.apply(sharpe),
    "total_return": cum.iloc[-1] - 1,
    "annualized_vol": strategies.std() * np.sqrt(365),
    "max_drawdown": cum.apply(max_drawdown),
    "n_trades": pd.Series({
        "pure_mean_reversion": (position_pure_mr.diff().abs() > 0).sum(),
        "trend_filtered_mean_reversion": (position_trend_filtered.diff().abs() > 0).sum(),
    }),
})
summary


## 7. Market-Neutrality Check


In [ ]:
btc_ret = np.log(df["btc_usd"]).diff()

exposure_check = pd.DataFrame({
    "pure_mr_vs_btc": [strategies["pure_mean_reversion"].corr(btc_ret)],
    "trend_filtered_vs_btc": [strategies["trend_filtered_mean_reversion"].corr(btc_ret)],
}, index=["correlation_with_btc_return"])
exposure_check


## 8. Does the Gross Edge Survive Real Trading Costs?


In [ ]:
TAKER_FEE_PER_FILL = 0.0005
FILLS_PER_ROUND_TRIP = 4

fee_drag = summary["n_trades"] * FILLS_PER_ROUND_TRIP * TAKER_FEE_PER_FILL
net_return_estimate = summary["total_return"] - fee_drag

cost_comparison = pd.DataFrame({
    "gross_total_return": summary["total_return"],
    "est_fee_drag": fee_drag,
    "est_net_total_return": net_return_estimate,
})
cost_comparison


## 9. Takeaways & Next Steps

- **Check Section 4 first**: like the stETH notebook, the collateral-demand hypothesis has a real
  documented precedent (the 2023 custody controversy), but this sample postdates it — a weak
  correlation here doesn't refute the mechanism, it just means this particular ~2-year window may not
  contain another comparable stress event.
- **The Coinbase delisting is real, independently-verified context**: worth checking whether other
  major venues have also reduced WBTC support before assuming deep, liquid two-sided markets exist
  for both legs of this trade at real size.
- **Kraken's OHLC endpoint caps history at ~720 daily candles**: confirmed empirically. For a longer
  backtest you'd need a different data source (a paid historical-data vendor, or another exchange
  with deeper retention) — this notebook's ~2-year window is the practical ceiling for this specific
  free approach.
- **Real-world execution isn't modeled**: WBTC redemption goes through a KYC'd merchant process with
  practical minimums, not an instant, permissionless mechanism — a real arbitrageur's actual
  achievable size and speed differ meaningfully from this backtest's assumptions.
- **Robustness**: validate out-of-sample and account for transaction costs (Section 8) before
  treating any of this as an actual edge.
